In [1]:
%cd drive/MyDrive/Thesis/Models/PSSM_NN

/content/drive/MyDrive/Thesis/Models/PSSM_NN


In [2]:
!ls

inference.ipynb  model.py  __pycache__	saved_model  training.ipynb


In [3]:
import numpy as np
import torch
import torch.nn as nn

from model import CNN_BRNN

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CNN_BRNN()
model = model.to(device)
model.load_state_dict(torch.load('./saved_model', map_location = torch.device(device)))
model.float()
model.eval()

CNN_BRNN(
  (conv1): Sequential(
    (0): Conv1d(21, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): ReLU()
  )
  (conv2): Sequential(
    (0): Conv1d(21, 64, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): ReLU()
  )
  (conv3): Sequential(
    (0): Conv1d(21, 64, kernel_size=(11,), stride=(1,), padding=(5,))
    (1): ReLU()
  )
  (brnn): GRU(192, 256, num_layers=3, batch_first=True, dropout=0.5, bidirectional=True)
  (fc): Sequential(
    (0): Linear(in_features=704, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=3, bias=True)
  )
)

In [5]:
PSSM_archive = np.load('../../Data/PSSMs2.npz')
# PSSM_archive = np.load('../../Data/PSSMs.npz')

In [7]:
def get_prediction(np_array_input):
  input = torch.from_numpy(np_array_input)
  input = torch.nn.functional.pad(input , (700 - input.shape[1], 0)).to(dtype=torch.float, device=device)
  input.unsqueeze_(0)
  
  output = model(input)
  _, prediction = torch.max(output, 1)
  prediction = prediction[:, 700 - np_array_input.shape[1] : ]
  prediction = prediction.numpy(force=True).astype('int8')
  assert prediction.shape[1] == np_array_input.shape[1], print(prediction.shape)
  prediction = prediction.squeeze()
  return(prediction)

def get_long_prediction(np_array_input):
  predictions = []
  length = np_array_input.shape[1]
  slices = length // 700 
  for i in range(slices):
    input = torch.from_numpy(np_array_input[ : , 700 * i : 700 * (i+1)]).to(dtype=torch.float, device=device)
    input.unsqueeze_(0)
    output= model(input)
    _, prediction = torch.max(output, 1)
    prediction = prediction.numpy(force=True)
    predictions.append(prediction)
  
  input = torch.from_numpy(np_array_input[ : , -700:]).to(dtype=torch.float, device=device)
  input.unsqueeze_(0)
  output = model(input)
  _, prediction = torch.max(output, 1)
  prediction = prediction.numpy(force=True)
  prediction = prediction[ : , -length%700:]
  predictions.append(prediction)

  predictions = np.concatenate(predictions, axis=None, dtype='int8')
  predictions = predictions.squeeze()
  assert len(predictions) == length, print(len(predictions), length)
  return(predictions)

In [8]:
PSSM_predictions = {}
for chain in PSSM_archive.files:
  try:
    if PSSM_archive[chain].shape[1] > 700:
      prediction = get_long_prediction(PSSM_archive[chain])
    else:
      prediction = get_prediction(PSSM_archive[chain])
    PSSM_predictions[chain] = prediction

  except:
    print(chain)

np.savez('../../Data/p1217.npz', **PSSM_predictions)

In [ ]:
# def get_batch(batch_IDs):
#   inputs = []
#   targets= []
#   for chain in batch_IDs:
#     target = torch.from_numpy(test_set[chain][1])
#     target = torch.nn.functional.pad(target, (700 - 1 - target.shape[-1],1), value = -9)
#     targets.append(target)

#     input = torch.from_numpy(PSSM_archive[chain])
#     input = torch.nn.functional.pad(input, (700 - 1 - input.shape[1],1))
#     inputs.append(input)
  
#   #stack the tensors
#   input_batch = torch.stack(inputs).to(device)
#   target_batch= torch.stack(targets).to(device)

#   return(input_batch, target_batch)

# bins = len(test_IDs)//100
# for i in range(bins):
#   if i== bins-1:
#     batch_IDS = test_IDs[100*i:]
#   else:
#     batch_IDs = test_IDs[100*i : 100*(i+1)]

#   input_batch, target_batch = get_batch(batch_IDs)
#   input_batch = input_batch.to(device)

#   outputs = model(input_batch)

#   for ix, chain in enumerate(batch_IDs):